In [2]:
# ============================================================
#  PARALLEL FEATURE EXTRACTION — v4
#  Platform : Kaggle  |  GPU : T4 x2
#  Fix: drop tf.Graph() entirely (it kills eager mode).
#       Use model(batch, training=False) instead of predict().
#       Isolate models by building each in its own thread with
#       a threading.Lock so weight downloads don't race.
# ============================================================
 
import numpy as np
import os, time, json, warnings, threading
import tensorflow as tf
warnings.filterwarnings('ignore')
 
from concurrent.futures import ThreadPoolExecutor, as_completed
from tensorflow.keras.applications import MobileNetV2, VGG16, DenseNet121
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mobile
from tensorflow.keras.applications.vgg16        import preprocess_input as pre_vgg
from tensorflow.keras.applications.densenet     import preprocess_input as pre_densenet
from tensorflow.keras.preprocessing import image
 
# ── Paths ────────────────────────────────────────────────────
TRAIN_DIR  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR   = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16          # safe per-thread batch size
 
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)}")
for g in gpus:
    print(" ", g)
 
# Eager mode must stay ON — never call tf.compat.v1.disable_eager_execution()
print(f"Eager mode: {tf.executing_eagerly()}")
 
# Lock so only one thread downloads/loads weights at a time
# (after loading, inference runs fully in parallel)
_build_lock = threading.Lock()
 
# ── Load all images into RAM once ────────────────────────────
def load_images(data_dir):
    imgs, labels = [], []
    for cls in sorted(os.listdir(data_dir)):
        cls_path = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_path) or cls.startswith('.'):
            continue
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        for fname in files:
            try:
                img = image.load_img(os.path.join(cls_path, fname),
                                     target_size=IMG_SIZE)
                imgs.append(image.img_to_array(img))
                labels.append(cls)
            except Exception as e:
                print(f"  Skipping {fname}: {e}")
    return np.array(imgs, dtype=np.float32), np.array(labels)
 
# ── Inference in small batches using direct model call ───────
def predict_in_batches(model, imgs_raw, preprocess_fn, batch_size=BATCH_SIZE):
    """
    Use model(x, training=False) instead of model.predict().
    model.predict() internally builds a tf.data pipeline which
    requires eager mode — but that can conflict across threads.
    Direct __call__ works in eager mode on any thread safely.
    """
    all_features = []
    n = len(imgs_raw)
    for start in range(0, n, batch_size):
        end   = min(start + batch_size, n)
        batch = imgs_raw[start:end].copy()
        batch = preprocess_fn(batch)
        batch_tensor = tf.constant(batch)                      # to tensor
        feats = model(batch_tensor, training=False).numpy()    # direct call
        all_features.append(feats)
    return np.vstack(all_features)
 
# ── Worker function ───────────────────────────────────────────
def run_model(task_name, model_fn, preprocess_fn,
              train_imgs, train_labels,
              test_imgs,  test_labels):
    print(f"  [{task_name}] waiting for build lock...")
 
    # Serialize model building only (weight download/load is not thread-safe)
    with _build_lock:
        print(f"  [{task_name}] building model...")
        base = model_fn(
            weights='imagenet',
            include_top=False,
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        base.trainable = False
        gap   = tf.keras.layers.GlobalAveragePooling2D()(base.output)
        model = tf.keras.Model(inputs=base.input, outputs=gap)
        print(f"  [{task_name}] model ready — starting inference")
 
    # Inference runs fully in parallel (lock released)
    t0 = time.perf_counter()
    f_train = predict_in_batches(model, train_imgs, preprocess_fn)
    f_test  = predict_in_batches(model, test_imgs,  preprocess_fn)
    elapsed = time.perf_counter() - t0
 
    print(f"  [{task_name}] done | inference: {elapsed:.2f}s | "
          f"train {f_train.shape}  test {f_test.shape}")
 
    return {
        'name'   : task_name,
        'f_train': f_train,
        'f_test' : f_test,
        'elapsed': elapsed,   # inference time only (fair comparison)
    }
 
# ── Pre-load images ───────────────────────────────────────────
print("\nLoading images from disk...")
t0 = time.perf_counter()
train_imgs, train_labels = load_images(TRAIN_DIR)
test_imgs,  test_labels  = load_images(TEST_DIR)
print(f"  train: {train_imgs.shape}  test: {test_imgs.shape}")
print(f"  load time : {time.perf_counter()-t0:.1f}s")
print(f"  train RAM : {train_imgs.nbytes/1e9:.2f} GB")
print(f"  test  RAM : {test_imgs.nbytes/1e9:.2f} GB")
 
# ── Parallel execution ────────────────────────────────────────
tasks = [
    ('MobileNetV2', MobileNetV2, pre_mobile),
    ('VGG16',       VGG16,       pre_vgg),
    ('DenseNet121', DenseNet121, pre_densenet),
]
results = {}
 
print("\n" + "="*60)
print("  PARALLEL FEATURE EXTRACTION  (3 threads)")
print("="*60)
 
wall_start = time.perf_counter()
 
with ThreadPoolExecutor(max_workers=3) as executor:
    future_map = {
        executor.submit(
            run_model,
            name, fn, pre,
            train_imgs, train_labels,
            test_imgs,  test_labels
        ): name
        for name, fn, pre in tasks
    }
    for future in as_completed(future_map):
        res = future.result()
        results[res['name']] = res
 
parallel_total = time.perf_counter() - wall_start   # wall-clock total
 
# ── Feature fusion ────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
 
X_train = np.hstack([results['MobileNetV2']['f_train'],
                     results['VGG16']['f_train'],
                     results['DenseNet121']['f_train']])
X_test  = np.hstack([results['MobileNetV2']['f_test'],
                     results['VGG16']['f_test'],
                     results['DenseNet121']['f_test']])
 
print(f"\nFused shape — train: {X_train.shape}  test: {X_test.shape}")
 
le = LabelEncoder()
y_train = le.fit_transform(train_labels)
y_test  = le.transform(test_labels)
 
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
 
print("Training SVM...")
svm = SVC(kernel='rbf', C=10, gamma='scale',
          probability=True, random_state=42, class_weight='balanced')
svm.fit(X_train_sc, y_train)
y_pred = svm.predict(X_test_sc)
 
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')
 
# ── Load sequential timing for speedup calc ───────────────────
try:
    with open('/kaggle/working/sequential_timing.json') as f:
        seq = json.load(f)
    seq_total = seq['Sequential_Total']
    speedup   = seq_total / parallel_total
    eff       = speedup / 3 * 100
    has_seq   = True
except FileNotFoundError:
    has_seq = False
    print("  Run sequential notebook first to get speedup ratio")
 
# ── Report ────────────────────────────────────────────────────
print("\n" + "="*60)
print("  TIMING REPORT")
print("="*60)
for name, _, _ in tasks:
    print(f"  {name:<14}: {results[name]['elapsed']:.2f}s  (inference)")
print(f"  {'─'*38}")
print(f"  PARALLEL WALL-CLOCK : {parallel_total:.2f}s")
if has_seq:
    print(f"  SEQUENTIAL TOTAL    : {seq_total:.2f}s")
    print(f"  SPEEDUP             : {speedup:.2f}x")
    print(f"  EFFICIENCY          : {eff:.1f}%")
 
print("\n" + "="*60)
print("  CLASSIFICATION RESULTS")
print("="*60)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  F1-Score  : {f1*100:.2f}%")
print("="*60)
 
# ── Save ──────────────────────────────────────────────────────
out = {
    'MobileNetV2_inference': results['MobileNetV2']['elapsed'],
    'VGG16_inference'      : results['VGG16']['elapsed'],
    'DenseNet121_inference': results['DenseNet121']['elapsed'],
    'Parallel_Wall_Clock'  : round(parallel_total, 4),
    'Speedup'              : round(speedup, 4) if has_seq else 'N/A',
    'Efficiency_pct'       : round(eff, 2)     if has_seq else 'N/A',
    'Accuracy'             : round(acc,  4),
    'Precision'            : round(prec, 4),
    'Recall'               : round(rec,  4),
    'F1'                   : round(f1,   4),
}
with open('/kaggle/working/parallel_timing.json', 'w') as f:
    json.dump(out, f, indent=2)
print("\n✅ Saved to /kaggle/working/parallel_timing.json")

GPUs detected: 2
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
Eager mode: True

Loading images from disk...
  train: (5600, 224, 224, 3)  test: (1600, 224, 224, 3)
  load time : 27.9s
  train RAM : 3.37 GB
  test  RAM : 0.96 GB

  PARALLEL FEATURE EXTRACTION  (3 threads)
  [MobileNetV2] waiting for build lock...
  [MobileNetV2] building model...
  [VGG16] waiting for build lock...
  [DenseNet121] waiting for build lock...
  [MobileNetV2] model ready — starting inference
  [VGG16] building model...


I0000 00:00:1776539108.304819  259458 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [VGG16] model ready — starting inference
  [DenseNet121] building model...
  [DenseNet121] model ready — starting inference
  [VGG16] done | inference: 55.41s | train (5600, 512)  test (1600, 512)
  [MobileNetV2] done | inference: 184.37s | train (5600, 1280)  test (1600, 1280)
  [DenseNet121] done | inference: 301.35s | train (5600, 1024)  test (1600, 1024)

Fused shape — train: (5600, 2816)  test: (1600, 2816)
Training SVM...

  TIMING REPORT
  MobileNetV2   : 184.37s  (inference)
  VGG16         : 55.41s  (inference)
  DenseNet121   : 301.35s  (inference)
  ──────────────────────────────────────
  PARALLEL WALL-CLOCK : 312.30s
  SEQUENTIAL TOTAL    : 2025.84s
  SPEEDUP             : 6.49x
  EFFICIENCY          : 216.2%

  CLASSIFICATION RESULTS
  Accuracy  : 93.88%
  Precision : 94.45%
  Recall    : 93.88%
  F1-Score  : 93.69%

✅ Saved to /kaggle/working/parallel_timing.json
